# Bayesian Optimisation: Directed Cell Migration

Uses the `BOptGPAX` inter-phase agent to find the optimal optogenetic
stimulation parameters that maximise directed cell migration.

**Steerable parameters (BO optimises):**
- `stim_cell_percentage` -- fraction of cell area to stimulate (0.1--0.5)
- `stim_exposure` -- stimulation light exposure in ms (0--500)

**Covariate (observed, not controlled):**
- `optocheck_mean_intensity` -- Cyan channel intensity at the optocheck frame

**Objective:** maximise `displacement` (Euclidean distance between baseline
and end-of-stimulation median positions)

In [ ]:
import os
import time
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ["JAX_LOG_COMPILES"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import jax

jax.config.update("jax_log_compiles", False)
for _name in list(logging.Logger.manager.loggerDict):
    if "jax" in _name or "absl" in _name:
        logging.getLogger(_name).setLevel(logging.WARNING)

from rtm_pymmcore.core.data_structures import (
    Channel,
    RTMSequence,
    SegmentationMethod,
)
from rtm_pymmcore.core.controller import Controller
from rtm_pymmcore.core.pipeline import ImageProcessingPipeline
from rtm_pymmcore.agents.bo_optimization import (
    BOptGPAX,
    BO_Parameter,
    BO_Objective,
    BO_Covariate,
)

## Microscope & pipeline setup

In [ ]:
# --- Microscope ---
# Option A: Simulated (requires `pip install virtual-microscope`)
from virtual_microscope.backends.optogenetic import setup_optogenetic
from rtm_pymmcore.microscope.simulation import UniMMCoreSimulation

core, sim = setup_optogenetic()
mic = UniMMCoreSimulation(mmc=core)

# Option B: Real microscope (uncomment and adapt)
# from rtm_pymmcore.microscope.pymmcore import PyMMCoreMicroscope
# mic = PyMMCoreMicroscope(config_file="path/to/config.cfg")

# --- Segmentation ---
from rtm_pymmcore.segmentation.base import OtsuSegmentator

# --- Feature extraction & tracking ---
from rtm_pymmcore.feature_extraction.simple import SimpleFE
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy

In [ ]:
# --- Experiment parameters ---
TIME_BETWEEN_TIMESTEPS = 15  # seconds between frames
N_FRAMES_BASELINE = 12  # 3 min baseline
FIRST_FRAME_STIM = N_FRAMES_BASELINE
LAST_FRAME_STIM = 120  # 27 min stimulation (108 frames)
N_FRAMES = 121  # baseline + stim + optocheck
PAUSE_BETWEEN_ITERATIONS = 15 * 60  # 15 min pause

# --- Storage ---
path = os.path.join(os.getcwd(), "output_bo_migration")

# --- Channels (use Channel dataclass instances) ---
img_channels = (Channel(config="FITC", exposure=300),)
stim_channel = Channel(config="CyanStim", exposure=500)
ref_channel = Channel(config="Cyan", exposure=500)  # optocheck

# --- Pipeline ---
pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=[
        SegmentationMethod(
            "labels", OtsuSegmentator(), use_channel=0, save_tracked=True
        )
    ],
    feature_extractor=SimpleFE("labels"),
    tracker=TrackerTrackpy(search_range=50),
    # stimulator=StimDirectedPercentage(),  # your stim engine
)

## Define the BO agent subclass

- `_create_events_for_cycle` -- builds `RTMEvent` objects with stim_exposure overridden per BO iteration
- `_preprocess_results` -- computes migration displacement from tracking data

In [ ]:
class MigrationBO(BOptGPAX):
    """BO agent for directed cell migration optimisation."""

    def __init__(
        self,
        *,
        fovs,
        n_frames,
        first_frame_stim,
        last_frame_stim,
        time_between_timesteps,
        img_channels,
        stim_channel,
        ref_channel,
        stage_positions,
        pause_between_iterations=0,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self._fovs = fovs
        self.n_frames = n_frames
        self.first_frame_stim = first_frame_stim
        self.last_frame_stim = last_frame_stim
        self.time_between_timesteps = time_between_timesteps
        self.img_channels = img_channels
        self.stim_channel = stim_channel
        self.ref_channel = ref_channel
        self.stage_positions = stage_positions
        self.pause_between_iterations = pause_between_iterations
        self._phase_counter = 0

    def _create_events_for_cycle(self, parameters: dict) -> list:
        """Build RTMEvents, overriding stim_exposure from BO."""
        phase_id = self._phase_counter

        # Override stim exposure from BO parameter
        stim_ch = Channel(
            config=self.stim_channel.config,
            exposure=parameters["stim_exposure"],
        )

        acq = RTMSequence(
            time_plan={"interval": self.time_between_timesteps, "loops": self.n_frames},
            stage_positions=self.stage_positions,
            channels=self.img_channels,
            stim_channels=(stim_ch,),
            stim_frames=range(self.first_frame_stim, self.last_frame_stim),
            ref_channels=(self.ref_channel,),
            ref_frames={self.n_frames - 1},  # optocheck at last frame
            rtm_metadata={
                "phase_name": f"BO_iter_{phase_id}",
                "phase_id": phase_id,
                "stim_cell_percentage": parameters["stim_cell_percentage"],
                "stim_exposure": parameters["stim_exposure"],
            },
        )
        return list(acq)

    def _preprocess_results(self, fov_tracks: dict) -> pd.DataFrame:
        """Compute migration displacement from tracking data."""
        if self._phase_counter > 0 and self.pause_between_iterations > 0:
            print(f"  Pausing {self.pause_between_iterations / 60:.0f} min...")
            time.sleep(self.pause_between_iterations)

        phase_id = self._phase_counter
        self._phase_counter += 1

        results = []
        for fov_idx, df_tracks in fov_tracks.items():
            if df_tracks.empty or "particle" not in df_tracks.columns:
                continue

            for particle_id, grp in df_tracks.groupby("particle"):
                grp = grp.sort_values("timestep")

                baseline = grp[grp["timestep"] < 4]
                end = grp[grp["timestep"] >= self.n_frames - 4]

                if len(baseline) < 3 or len(end) < 3:
                    continue

                baseline_area = baseline["area"].median()
                if baseline_area < 50:
                    continue

                displacement = np.sqrt(
                    (end["x"].median() - baseline["x"].median()) ** 2
                    + (end["y"].median() - baseline["y"].median()) ** 2
                )

                optocheck = 0.0
                if "optocheck_mean_intensity" in grp.columns:
                    vals = grp["optocheck_mean_intensity"].dropna()
                    if not vals.empty:
                        optocheck = float(vals.median())

                stim_pct = (
                    grp["stim_cell_percentage"].iloc[0]
                    if "stim_cell_percentage" in grp.columns
                    else 0
                )
                stim_exp = (
                    grp["stim_exposure"].iloc[0]
                    if "stim_exposure" in grp.columns
                    else 0
                )

                results.append(
                    {
                        "stim_cell_percentage": stim_pct,
                        "stim_exposure": stim_exp,
                        "optocheck_mean_intensity": optocheck,
                        "displacement": displacement,
                    }
                )

        if not results:
            print(f"Warning: no valid cells in phase {phase_id}")
            return pd.DataFrame()

        df = pd.DataFrame(results)
        print(
            f"  Phase {phase_id}: {len(df)} cells, "
            f"mean displacement={df['displacement'].mean():.2f} px"
        )
        return df

## Configure and run Bayesian Optimisation

In [ ]:
# --- FOV positions ---
stage_positions = [(0.0, 0.0, 0.0)]  # replace with your FOV positions

# --- BO configuration ---
bo_params = [
    BO_Parameter(name="stim_cell_percentage", bounds=(0.1, 0.5), spacing=0.1),
    BO_Parameter(name="stim_exposure", bounds=(0.0, 500.0), spacing=100.0),
]
bo_covariates = [BO_Covariate(name="optocheck_mean_intensity")]
bo_objective = BO_Objective(name="displacement", goal="maximize")

# --- Create agent ---
agent = MigrationBO(
    storage_path=path,
    fovs=list(range(len(stage_positions))),
    n_frames=N_FRAMES,
    first_frame_stim=FIRST_FRAME_STIM,
    last_frame_stim=LAST_FRAME_STIM,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    img_channels=img_channels,
    stim_channel=stim_channel,
    ref_channel=ref_channel,
    stage_positions=stage_positions,
    pause_between_iterations=PAUSE_BETWEEN_ITERATIONS,
    parameters_to_optimize=bo_params,
    objective_metric=bo_objective,
    bo_covariates=bo_covariates,
    n_iterations=10,
    acquisition_function="ei",
    n_cov_samples=30,
    ei_xi=0.2,
    ei_xi_final=0.01,
    ei_xi_decay_fraction=0.7,
    verbose=True,
)
agent.add_fovs(list(range(len(stage_positions))))

# --- Create controller and register agent ---
ctrl = Controller(mic, pipeline, agent=agent)

# --- Run the BO loop ---
agent.run()

## Visualise results

In [ ]:
if agent.x is not None and agent.y is not None:
    x_data = np.array(agent.x)
    y_data = np.array(agent.y).ravel()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sc = axes[0].scatter(
        x_data[:, 0],
        x_data[:, 1],
        c=y_data,
        cmap="viridis",
        s=40,
        edgecolors="k",
        linewidths=0.5,
    )
    axes[0].set_xlabel("stim_cell_percentage")
    axes[0].set_ylabel("stim_exposure (ms)")
    axes[0].set_title("Measured displacement")
    fig.colorbar(sc, ax=axes[0], label="displacement (px)")

    cumulative_best = np.maximum.accumulate(y_data)
    axes[1].plot(cumulative_best, "o-")
    axes[1].set_xlabel("Measurement index")
    axes[1].set_ylabel("Best displacement so far (px)")
    axes[1].set_title("BO convergence")

    plt.tight_layout()
    plt.show()
else:
    print("No data collected yet.")

In [ ]:
# 3D visualization of the GP-learned landscape
import gpax

rng_key, rng_key_pred = gpax.utils.get_keys()

pct_vals = np.linspace(0.1, 0.5, 15)
exp_vals = np.linspace(0.0, 500.0, 15)
opto_obs = agent.df_results["optocheck_mean_intensity"].values
opto_vals = np.linspace(opto_obs.min(), opto_obs.max(), 10)

grid = np.array([[p, e, o] for p in pct_vals for e in exp_vals for o in opto_vals])
grid_scaled = agent._x_scaler.transform(grid)
y_pred_scaled, _ = agent.model.predict(rng_key_pred, grid_scaled, noiseless=True)
y_pred = agent._y_scaler.inverse_transform(y_pred_scaled).flatten()

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(
    grid[:, 0],
    grid[:, 1],
    grid[:, 2],
    c=np.asarray(y_pred),
    cmap="viridis",
    s=12,
    alpha=0.6,
)
ax.set_xlabel("stim_cell_percentage")
ax.set_ylabel("stim_exposure (ms)")
ax.set_zlabel("optocheck")
ax.set_title("GP-predicted displacement landscape")
fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.12, label="displacement (px)")

opt_idx = int(np.argmax(np.asarray(y_pred)))
ax.scatter(
    grid[opt_idx, 0],
    grid[opt_idx, 1],
    grid[opt_idx, 2],
    c="red",
    s=300,
    marker="*",
    edgecolors="black",
    linewidths=2,
    label=f"Optimum: pct={grid[opt_idx,0]:.2f}, exp={grid[opt_idx,1]:.0f}ms",
    zorder=5,
)
ax.legend()
plt.tight_layout()
plt.show()

print(f"GP-predicted optimum:")
print(f"  stim_cell_percentage = {grid[opt_idx, 0]:.2f}")
print(f"  stim_exposure = {grid[opt_idx, 1]:.0f} ms")
print(f"  predicted displacement = {float(y_pred[opt_idx]):.2f} px")